In [2]:
import pandas as pd
import numpy as np

In [3]:
movies_table = pd.read_csv('../data/movies.csv')

## Парсинг данных

In [4]:
from bs4 import BeautifulSoup
import requests

/Users/irinaaristova/Documents/Programming/ml-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:

from bs4 import BeautifulSoup
import requests

url = "http://www.usinflationcalculator.com/inflation/consumer-price-index-and-annual-percent-changes-from-1913-to-2008/"
r = requests.get(url)
data = r.text
soup = BeautifulSoup(data, 'html.parser')

table = soup.find('table')
rows = table.tbody.find_all('tr')

years = []
cpis = []

for row in rows:
    cells = row.find_all('td')
    if len(cells) > 0:
        year = cells[0].get_text()
        if year.isdigit() and int(year) < 2017:
            years.append(int(year))
            cpi_val = cells[13].get_text().replace(',', '.') # Замена запятой на точку для float
            cpis.append(float(cpi_val))

cpi_table = pd.DataFrame({"year": years, "avg_annual_cpi": cpis})
cpi_table.head()

cpi_table.to_csv("../data/Practice2_Аристова_CPI.csv", index=False)


In [6]:
def get_real_value(nominal_amt, old_cpi, new_cpi):
    real_value = (nominal_amt * new_cpi) / old_cpi
    return real_value

CPI_2016 = float(cpi_table[cpi_table['year'] == 2016]['avg_annual_cpi'].values[0])

real_domestic_gross = []
real_budget_values = []


for index, row in movies_table.iterrows():
    gross = row['gross']
    budget = row['budget']
    year = row['title_year']

    if pd.isna(year):
        real_domestic_gross.append(np.nan)
        real_budget_values.append(np.nan)
        continue

    cpi_row = cpi_table[cpi_table['year'] == int(year)]

    if cpi_row.empty:
        real_domestic_gross.append(np.nan)
        real_budget_values.append(np.nan)
        continue

    cpi = float(cpi_row['avg_annual_cpi'].values[0])

    real_gross = get_real_value(gross, cpi, CPI_2016)
    real_budget = get_real_value(budget, cpi, CPI_2016)

    real_domestic_gross.append(real_gross)
    real_budget_values.append(real_budget)

movies_table["real_domestic_gross"] = real_domestic_gross
movies_table["real_budget"] = real_budget_values

profits = []
roi_vals = []

for index, row in movies_table.iterrows():
    profit = row['real_domestic_gross'] - row['real_budget']
    budget = row['real_budget']
    num = profit - budget
    den = budget

    roi = (num / den) * 100

    profits.append(profit)
    roi_vals.append(roi)

movies_table['profit'] = profits
movies_table['roi'] = roi_vals

movies_table.head()

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;,real_domestic_gross,real_budget,profit,roi
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,237000000.0,2009.0,936.0,7.9,1.78,33000;,8.507937e+08,2.651368e+08,5.856569e+08,120.888543
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,300000000.0,2007.0,5000.0,7.1,2.35,0;,3.582208e+08,3.473329e+08,1.088790e+07,-96.865283
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,245000000.0,2015.0,393.0,6.8,2.35,85000;,2.025981e+08,2.480907e+08,-4.549257e+07,-118.337071
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,250000000.0,2012.0,23000.0,8.5,2.35,164000;,4.684551e+08,2.613385e+08,2.071167e+08,-20.747743
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,12.0,7.1,NaN,0;,NaN,NaN,NaN,NaN


## Исследование исходных данных 
### Вывод базовой информации

In [7]:
movies_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   color                       5025 non-null   object 
 1   Director_Name               4872 non-null   object 
 2   num_Critic_for_reviews      4927 non-null   float64
 3   duration                    4959 non-null   float64
 4   director_Facebook_likes     4872 non-null   float64
 5   actor_3_Facebook_likes      4953 non-null   float64
 6   actor_2_name                4963 non-null   object 
 7   Actor_1_Facebook_likes      4968 non-null   float64
 8   gross                       4104 non-null   float64
 9   genres                      4974 non-null   object 
 10  actor_1_name                4968 non-null   object 
 11  movie_Title                 4974 non-null   object 
 12  num_voted_users             4974 non-null   float64
 13   cast_total_facebook_likes  4974 

<font size=3>
Датасет содержит следующие поля:

* `color` — цвет фильма (цветной или чёрно-белый)
* `Director_Name` — имя режиссёра
* `num_Critic_for_reviews` — количество отзывов кинокритиков
* `duration` — продолжительность фильма (в минутах)
* `director_Facebook_likes` — количество лайков режиссёра в Facebook
* `actor_3_Facebook_likes` — количество лайков третьего актёра в Facebook
* `actor_2_name` — имя второго актёра
* `Actor_1_Facebook_likes` — количество лайков первого актёра в Facebook
* `gross` — кассовые сборы (номинальные доллары)
* `genres` — жанры фильма
* `actor_1_name` — имя первого актёра
* `movie_Title` — название фильма
* `num_voted_users` — количество пользователей, проголосовавших на IMDb
* `cast_total_facebook_likes` — суммарное количество лайков актёрского состава в Facebook
* `actor_3_name` — имя третьего актёра
* `facenumber_in_poster` — количество лиц на постере фильма
* `plot_keywords` — ключевые слова сюжета
* `movie_imdb_link` — ссылка на страницу фильма на IMDb
* `num_user_for_reviews` — количество пользовательских отзывов
* `language` — язык фильма
* `country` — страна производства
* `content_rating` — возрастной рейтинг (например, PG-13, R)
* `budget` — бюджет фильма (номинальные доллары)
* `title_year` — год выхода фильма
* `actor_2_facebook_likes` — количество лайков второго актёра в Facebook
* `imdb_score` — рейтинг фильма на IMDb
* `aspect_ratio` — соотношение сторон экрана
* `movie_facebook_likes;` — количество лайков фильма в Facebook
* `real_domestic_gross` — кассовые сборы, приведённые к реальным долларам 2016 года
* `real_budget` — бюджет, приведённый к реальным долларам 2016 года
* `profit` — прибыль (разница между реальными сборами и реальным бюджетом)
* `roi` — рентабельность инвестиций (Return on Investment), выраженная в процентах

</font>


In [8]:
movies_table.head(20)

,color,Director_Name,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,actor_2_name,Actor_1_Facebook_likes,gross,genres,...,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,movie_facebook_likes;,real_domestic_gross,real_budget,profit,roi
0,Color,James Cameron,723.0,178.0,0.0,855.0,Joel David Moore,1000.0,760505847.0,Action|Adventure|Fantasy|Sci-Fi,...,237000000.0,2009.0,936.0,7.9,1.78,33000;,8.507937e+08,2.651368e+08,5.856569e+08,120.888543
1,Colour,Gore Verbinski,302.0,169.0,563.0,1000.0,Orlando Bloom,40000.0,309404152.0,Action|Adventure|Fantasy,...,300000000.0,2007.0,5000.0,7.1,2.35,0;,3.582208e+08,3.473329e+08,1.088790e+07,-96.865283
2,Colour,Sam Mendes,602.0,148.0,0.0,161.0,Rory Kinnear,11000.0,200074175.0,Action|Adventure|Thriller,...,245000000.0,2015.0,393.0,6.8,2.35,85000;,2.025981e+08,2.480907e+08,-4.549257e+07,-118.337071
3,Color,Christopher Nolan,813.0,164.0,22000.0,23000.0,Christian Bale,27000.0,448130642.0,Action|Thriller,...,250000000.0,2012.0,23000.0,8.5,2.35,164000;,4.684551e+08,2.613385e+08,2.071167e+08,-20.747743
4,NaN,Doug Walker,NaN,NaN,131.0,NaN,Rob Walker,131.0,NaN,Documentary,...,NaN,NaN,12.0,7.1,NaN,0;,NaN,NaN,NaN,NaN
5,Colour,Andrew Stanton,462.0,132.0,475.0,530.0,Samantha Morton,640.0,73058679.0,Action|Adventure|Sci-Fi,...,263700000.0,2012.0,632.0,6.6,2.35,24000;,7.637218e+07,2.756598e+08,-1.992877e+08,-172.294775
6,Color,Sam Raimi,392.0,156.0,0.0,4000.0,James Franco,24000.0,336530303.0,Action|Adventure|Romance,...,258000000.0,2007.0,11000.0,6.2,2.35,0;,3.896268e+08,2.987063e+08,9.092051e+07,-69.561898
7,Color,Nathan Greno,324.0,100.0,15.0,284.0,Donna Murphy,799.0,200807262.0,Adventure|Animation|Comedy|Family|Fantasy|Musi...,...,260000000.0,2010.0,553.0,7.8,1.85,29000;,2.210219e+08,2.861734e+08,-6.515148e+07,-122.766438
8,Colour,Joss Whedon,635.0,141.0,0.0,19000.0,Robert Downey Jr.,26000.0,458991599.0,Action|Adventure|Sci-Fi,...,250000000.0,2015.0,21000.0,7.5,2.35,118000;,4.647818e+08,2.531538e+08,2.116281e+08,-16.403360
9,Colour,David Yates,375.0,153.0,282.0,10000.0,Daniel Radcliffe,25000.0,301956980.0,Adventure|Family|Fantasy|Mystery,...,250000000.0,2009.0,11000.0,7.5,2.35,10000;,3.378055e+08,2.796802e+08,5.812535e+07,-79.217208


In [9]:
movies_table.describe()

,num_Critic_for_reviews,duration,director_Facebook_likes,actor_3_Facebook_likes,Actor_1_Facebook_likes,gross,num_voted_users,cast_total_facebook_likes,facenumber_in_poster,num_user_for_reviews,budget,title_year,actor_2_facebook_likes,imdb_score,aspect_ratio,real_domestic_gross,real_budget,profit,roi
count,4927.000000,4959.000000,4872.000000,4953.000000,4968.000000,4.104000e+03,4.974000e+03,4974.000000,4961.000000,4956.000000,4.494000e+03,4869.000000,4963.000000,4974.000000,4654.000000,4.101000e+03,4.487000e+03,3.845000e+03,3845.000000
mean,140.572965,107.193991,691.233990,647.790430,6573.525765,4.862765e+07,8.382561e+04,9729.287495,1.372707,273.584746,3.998652e+07,2002.487985,1660.086641,6.439767,2.223350,6.989709e+07,5.142079e+07,1.570334e+07,429.665140
std,121.858265,24.977939,2822.022077,1672.724849,15077.147622,6.853339e+07,1.386775e+05,18228.145064,2.017257,378.750005,2.073754e+08,12.433410,4055.957071,1.124002,1.393249,1.316657e+08,2.520962e+08,2.958195e+08,13039.409335
min,1.000000,7.000000,0.000000,0.000000,0.000000,1.620000e+02,5.000000e+00,0.000000,0.000000,1.000000,2.180000e+02,1916.000000,0.000000,1.600000,1.180000,1.642384e+02,2.843561e+02,-1.454007e+10,-199.998200
25%,50.000000,93.000000,7.000000,135.000000,617.000000,5.480826e+06,8.688250e+03,1430.250000,0.000000,65.000000,6.000000e+06,1999.000000,284.000000,5.800000,1.850000,7.045993e+06,9.658268e+06,-1.351505e+07,-154.703070
50%,110.000000,103.000000,49.000000,372.000000,989.000000,2.559138e+07,3.450400e+04,3097.500000,1.000000,157.000000,2.000000e+07,2005.000000,595.000000,6.600000,2.350000,3.336372e+07,2.667468e+07,1.188396e+06,-93.333333
75%,195.000000,118.000000,197.000000,636.000000,11000.000000,6.241428e+07,9.646275e+04,13808.750000,2.000000,327.000000,4.500000e+07,2011.000000,919.000000,7.200000,2.350000,8.365621e+07,6.067329e+07,3.350358e+07,22.740068
max,813.000000,511.000000,23000.000000,23000.000000,640000.000000,7.605058e+08,1.689764e+06,656730.000000,43.000000,5060.000000,1.221550e+10,2016.000000,137000.000000,9.500000,16.000000,3.430119e+09,1.454269e+10,3.361450e+09,719248.553333


### Обработка пропущенных значений

In [32]:
categorical_cols = movies_table.select_dtypes(include=['object']).columns

movies_table[categorical_cols[0]].value_counts()

color
Color                                                                                                                                                                                                                                                                                                                              4725
 Black and White                                                                                                                                                                                                                                                                                                                    207
Colour                                                                                                                                                                                                                                                                                                                               14
color     

In [11]:
(movies_table.isna().sum()).sort_values(ascending=False)

roi                           1198
profit                        1198
real_domestic_gross            942
gross                          939
real_budget                    556
budget                         549
aspect_ratio                   389
content_rating                 366
plot_keywords                  216
title_year                     174
director_Facebook_likes        171
Director_Name                  171
num_Critic_for_reviews         116
actor_3_Facebook_likes          90
actor_3_name                    90
num_user_for_reviews            87
duration                        84
facenumber_in_poster            82
language                        82
actor_2_name                    80
actor_2_facebook_likes          80
actor_1_name                    75
Actor_1_Facebook_likes          75
country                         72
movie_facebook_likes;           69
genres                          69
imdb_score                      69
movie_imdb_link                 69
 cast_total_facebook